In [5]:
import pandas as pd
import re
from collections import Counter

# 1. 데이터 로드 및 전처리
df_tsv = pd.read_csv(r'C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\Stability\S669.tsv', sep='\t')
df_csv = pd.read_csv(r'C:\Users\Kunny\Downloads\S669\S669\S669.csv')

def extract_num(mut_str):
    match = re.search(r'\d+', str(mut_str))
    return int(match.group()) if match else None

# CSV 미리 정리 (매칭 속도 향상)
df_csv['WT'] = df_csv['PDB_Mut'].str[0]
df_csv['MT'] = df_csv['PDB_Mut'].str[-1]
df_csv['PDB_POS_VAL'] = df_csv['PDB_Mut'].apply(extract_num)

df_tsv['PDB_ID_FINAL'] = None
df_tsv['PDB_POS_FINAL'] = None
df_tsv['OFFSET'] = None

# 2. 단백질별로 루프 실행
for protein_id in df_tsv['PDB'].unique():
    subset_tsv = df_tsv[df_tsv['PDB'] == protein_id]
    
    # 해당 단백질에 대응하는 CSV 데이터 찾기 (예: P47992 -> 1j8iA 등)
    # 여기서는 간단히 WT, MT, DDG가 겹치는 데이터가 있는 CSV 그룹을 찾습니다.
    # (실제로는 앞서 만든 mapping_results.json의 PDB ID를 활용하면 더 정확합니다)
    
    # 2-1. 먼저 해당 단백질의 가능한 모든 Offset 후보 계산
    offsets = []
    for _, t_row in subset_tsv.iterrows():
        matches = df_csv[
            (df_csv['WT'] == t_row['WT']) & 
            (df_csv['MT'] == t_row['MT']) & 
            (abs(df_csv['Experimental_DDG_dir'] - t_row['DDG']) < 0.01) # 소수점 오차 허용
        ]
        for _, c_row in matches.iterrows():
            offsets.append(t_row['POS'] - c_row['PDB_POS_VAL'])
    
    if not offsets:
        continue
        
    # 2-2. 가장 지배적인 Offset(최빈값) 결정
    common_offset = Counter(offsets).most_common(1)[0][0]
    
    # 2-3. 결정된 Offset을 바탕으로 정밀 매핑
    for idx in subset_tsv.index:
        t_row = df_tsv.loc[idx]
        # WT, MT, DDG가 맞으면서 Offset까지 완벽한 것 찾기
        final_match = df_csv[
            (df_csv['WT'] == t_row['WT']) & 
            (df_csv['MT'] == t_row['MT']) & 
            (abs(df_csv['Experimental_DDG_dir'] - t_row['DDG']) < 0.01) &
            ((t_row['POS'] - df_csv['PDB_POS_VAL']) == common_offset)
        ]
        
        if len(final_match) >= 1:
            df_tsv.at[idx, 'PDB_ID_FINAL'] = final_match.iloc[0]['Protein']
            df_tsv.at[idx, 'PDB_POS_FINAL'] = final_match.iloc[0]['PDB_POS_VAL']
            df_tsv.at[idx, 'OFFSET'] = common_offset
        else:
            print(f"⚠️ 매치 실패: {protein_id} POS {t_row['POS']} (일치하는 Offset {common_offset} 없음)")

# 3. 결과 확인
print("\n[매핑 결과 예시 - P47992]")
print(df_tsv[df_tsv['PDB'] == 'P47992'][['PDB', 'WT', 'POS', 'PDB_POS_FINAL', 'OFFSET']].head(10))

df_tsv.to_csv("S669_Offset_Corrected.tsv", sep='\t', index=False)

⚠️ 매치 실패: P78352 POS 332 (일치하는 Offset 0 없음)
⚠️ 매치 실패: P78352 POS 332 (일치하는 Offset 0 없음)

[매핑 결과 예시 - P47992]
        PDB WT  POS PDB_POS_FINAL OFFSET
191  P47992  K   46            25     21
192  P47992  K   87            66     21
193  P47992  R   44            23     21
194  P47992  R   56            35     21
195  P47992  R   64            43     21
196  P47992  R   30             9     21
